In [1]:
from textwrap import dedent

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.orm import Query

import src
from src.data.models import Channel
from src.data.models import Video

In [2]:
pd.set_option("display.max_rows", 256)
engine = create_engine(src.PS_ENGINE)

In [3]:
query = (
    Query(Video)
    .join(Channel)
    .filter(Video.is_valid == True, Video.format == "videos")
    .with_entities(
        Video.id,
        Video.title,
        Video.description,
        Video.duration,
        Video.datetime_upload,
        Channel.uploader_id,
    )
)

with engine.connect() as conn:
    df = pd.read_sql(query.statement, conn)


def build_url(video_id):
    return f"https://www.youtube.com/watch?v={video_id}"


def build_doccano_content(row):
    text = f"""
    {row["title"]}

    ----------------------------------------

    {row["description"]}
    """
    
    return dedent(text)

df["url"] = df["id"].apply(build_url)
df["doccano_text"] = df.apply(build_doccano_content, axis=1)

In [4]:
df = df.groupby("uploader_id").sample(60)

In [5]:
len(df)

480

In [6]:
df.drop(["title", "description"], axis=1).to_csv(
    src.PATH / "data/video_classification/unlabeled.csv", index=False,
)